# Phase 2 climat — Visualisation des résultats

Ce notebook visualise les résultats du pipeline phase 2 sur l'export du **12 mai 2026** (127 bulletins, 152 images).

## Note méthodologique

Le statut `CONFIRMED` indique simplement qu'un utilisateur a validé une suggestion algorithmique du moteur de règles ; il n'implique aucune validation experte. Le statut est donc traité ici comme une **information descriptive parmi d'autres**, et non comme un critère de qualité. Aucun bulletin n'est privilégié dans l'analyse.

## Sections

1. Setup et chargement des données
2. Entonnoir de couverture — *matplotlib*
3. Distribution du corpus — *matplotlib*
4. **Géographie** des 31 bulletins — *plotly interactif*
5. **Chronologie** des observations — *plotly interactif*
6. Distribution des 12 features météo — *matplotlib*
7. **Heatmap** bulletins × features — *plotly interactif*
8. **Tableau détaillé** des 31 bulletins avec leur contexte météo
9. Synthèse texte

⚠ Ce notebook **ne fait pas tourner `weather_rules.py`** — on regarde la donnée brute pour développer une intuition agronomique avant la classification.

## Dépendances

```bash
pip install pandas numpy matplotlib plotly
```

## 1. Setup et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Style matplotlib sobre, lisible en impression
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

# Palette cohérente (alignée avec le schéma du pipeline)
C_PRIMARY   = '#534AB7'  # violet
C_SECONDARY = '#0F6E56'  # teal
C_ACCENT    = '#BA7517'  # amber
C_WARNING   = '#993C1D'  # coral
C_NEUTRAL   = '#5F5E5A'  # gray
C_LIGHT     = '#D8D6F2'

OUTPUTS = Path('outputs')

def read_csv_if_exists(path, **kwargs):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path, **kwargs)
    print(f'[WARN] fichier absent : {path}')
    return pd.DataFrame()

df = pd.read_csv(OUTPUTS / 'phase2_bulletins_features.csv',
                 parse_dates=['datetime_observation'])

# Fichiers complémentaires utiles pour l'audit
exif = read_csv_if_exists(OUTPUTS / 'manifest_exif.csv')
context = read_csv_if_exists(OUTPUTS / 'phase2_context.csv',
                             parse_dates=['datetime_original'])
fetch_log = read_csv_if_exists(OUTPUTS / 'weather_fetch_log.csv')

print(f'{len(df)} bulletins chargés, {df.shape[1]} colonnes')
print(f'{len(context)} lignes image-context chargées')
print(f'{len(exif)} lignes EXIF chargées')
print(f'{len(fetch_log)} lignes log météo chargées')

In [ ]:
# Sous-ensembles utiles (sans distinction CONFIRMED/NOT_CONFIRMED)
# Les colonnes booléennes peuvent contenir des NaN pour les bulletins sans image.
bool_cols = ['phase2_ready', 'phase2_ready_loose', 'outdoor', 'status_confirmed']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype('boolean').fillna(False).astype(bool)

if 'phase2_ready' in context.columns:
    for col in ['phase2_ready', 'phase2_ready_loose', 'outdoor', 'status_confirmed', 'is_test']:
        if col in context.columns:
            context[col] = context[col].astype('boolean').fillna(False).astype(bool)

df_with_img    = df[df['n_images_found'].fillna(0) > 0]
df_ready       = df[df['phase2_ready']]
df_ready_loose = df[df['phase2_ready_loose']]
df_features    = df[df['rain_sum_15d'].notna()]

df_pi = df[df['type'] == 'PEST_IDENTIFICATION']
df_pr = df[df['type'] == 'PEST_REPORT']

print('Total bulletins              :', len(df))
print('  PEST_IDENTIFICATION        :', len(df_pi))
print('  PEST_REPORT                :', len(df_pr))
print('  avec images                :', len(df_with_img))
print('  phase2_ready (strict)      :', len(df_ready))
print('  phase2_ready_loose         :', len(df_ready_loose))
print('  avec features météo        :', len(df_features))

## 1bis. Contrôles de cohérence des fichiers intermédiaires

Cette section vérifie que les sorties des scripts précédents sont cohérentes avant d'interpréter les variables météo.

In [ ]:
checks = {
    'bulletins_features': len(df),
    'context_image_rows': len(context),
    'exif_rows': len(exif),
    'fetch_log_rows': len(fetch_log),
    'bulletins_with_features': int(df['rain_sum_15d'].notna().sum()),
    'images_with_features_loose': int(context['phase2_ready_loose'].sum()) if 'phase2_ready_loose' in context.columns else np.nan,
}

pd.DataFrame.from_dict(checks, orient='index', columns=['valeur'])

In [ ]:
# Cohérence n_images officiel vs images agrégées
cols = ['bulletin_id', 'n_images', 'n_images_found', 'n_images_match', 'n_images_with_exif',
        'date_span_minutes', 'phase2_ready', 'phase2_ready_loose']
audit_cols = [c for c in cols if c in df.columns]
df[audit_cols].describe(include='all')

## 1ter. Audit EXIF et cache météo

L'objectif est de documenter le principal résultat qualité de la phase 2 : les dates EXIF sont très disponibles, alors que la limite principale vient plutôt de la présence d'un jardin géolocalisé.

In [ ]:
if not exif.empty:
    display_cols = [c for c in [
        'filename', 'datetime_original', 'datetime_source',
        'has_exif_date', 'make', 'model', 'width', 'height', 'error'
    ] if c in exif.columns]
    print('Colonnes EXIF affichées :', display_cols)
    display(exif[display_cols].head())

    if 'datetime_original' in exif.columns:
        n_exif_date = exif['datetime_original'].notna().sum()
        print(f'Date EXIF exploitable : {n_exif_date} / {len(exif)} ({100*n_exif_date/len(exif):.1f} %)')

    if 'model' in exif.columns:
        print('\nTop modèles téléphone :')
        display(exif['model'].value_counts(dropna=False).head(12).rename_axis('model').reset_index(name='n'))

In [ ]:
if not fetch_log.empty:
    status_counts = fetch_log['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='n')
    display(status_counts)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(status_counts['status'].astype(str), status_counts['n'], color=C_PRIMARY)
    ax.set_title('Statut du fetch météo', loc='left', fontweight='bold')
    ax.set_ylabel('Nombre')
    for i, v in enumerate(status_counts['n']):
        ax.text(i, v + 0.3, str(v), ha='center')
    plt.tight_layout()
    plt.show()

## 2. Entonnoir de couverture (matplotlib)

Comment on passe de 127 bulletins officiels DJMA aux bulletins exploitables pour l'analyse météo.

In [ ]:
stages = [
    ('Bulletins officiels DJMA',     len(df)),
    ('Avec images sur disque',       (df['n_images_found'].fillna(0) > 0).sum()),
    ('Avec date EXIF',               df['datetime_observation'].notna().sum()),
    ('Avec coordonnées GPS',         df['latitude'].notna().sum()),
    ('Outdoor (strict)',             df['outdoor'].sum()),
    ('phase2_ready (strict)',        df['phase2_ready'].sum()),
    ('phase2_ready_loose',           df['phase2_ready_loose'].sum()),
]
labels, values = zip(*stages)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(labels)), values, color=C_PRIMARY, alpha=0.85)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel('Nombre de bulletins')
ax.set_title('Entonnoir de couverture phase 2', loc='left', fontweight='bold')

for i, (bar, v) in enumerate(zip(bars, values)):
    pct = 100 * v / values[0]
    ax.text(v + 1.5, i, f'{v}  ({pct:.1f}%)', va='center', fontsize=9)

ax.set_xlim(0, max(values) * 1.18)
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

## 3. Distribution du corpus (matplotlib)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 3a. Status (descriptif, pas une métrique de qualité)
status_counts = df['status'].value_counts(dropna=False)
axes[0, 0].bar(status_counts.index.astype(str), status_counts.values,
               color=[C_NEUTRAL, C_NEUTRAL])
axes[0, 0].set_title('Status utilisateur (descriptif)', loc='left', fontweight='bold')
axes[0, 0].set_ylabel('Nombre')
for i, v in enumerate(status_counts.values):
    axes[0, 0].text(i, v + 1, str(v), ha='center', fontsize=9)

# 3b. Type (PI vs PR)
type_counts = df['type'].value_counts(dropna=False)
axes[0, 1].bar(type_counts.index.astype(str), type_counts.values,
               color=[C_PRIMARY, C_ACCENT])
axes[0, 1].set_title('Type de tunnel', loc='left', fontweight='bold')
axes[0, 1].set_ylabel('Nombre')
for i, v in enumerate(type_counts.values):
    axes[0, 1].text(i, v + 1, str(v), ha='center', fontsize=9)

# 3c. plant_location
loc_counts = df['plant_location'].value_counts(dropna=False)
axes[1, 0].barh(loc_counts.index.astype(str), loc_counts.values, color=C_SECONDARY)
axes[1, 0].set_title('Plant location', loc='left', fontweight='bold')
axes[1, 0].set_xlabel('Nombre')
axes[1, 0].invert_yaxis()
for i, v in enumerate(loc_counts.values):
    axes[1, 0].text(v + 0.5, i, str(v), va='center', fontsize=9)

# 3d. n_images
n_img_counts = df['n_images'].value_counts().sort_index()
axes[1, 1].bar(n_img_counts.index.astype(str), n_img_counts.values, color=C_ACCENT)
axes[1, 1].set_title("Nombre d'images par bulletin", loc='left', fontweight='bold')
axes[1, 1].set_xlabel("n_images")
axes[1, 1].set_ylabel('Nombre de bulletins')
for i, v in enumerate(n_img_counts.values):
    axes[1, 1].text(i, v + 0.5, str(v), ha='center', fontsize=9)

plt.suptitle('Caractérisation du corpus (127 bulletins)',
             fontsize=12, fontweight='bold', x=0.07, y=1.00, ha='left')
plt.tight_layout()
plt.show()

In [ ]:
# Top bioagresseurs et espèces
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bio = df['nom_bioagresseur'].value_counts().head(10)
axes[0].barh(bio.index, bio.values, color=C_WARNING)
axes[0].set_title('Top 10 bioagresseurs suggérés (bulletins avec diagnostic)',
                  loc='left', fontweight='bold')
axes[0].invert_yaxis()
axes[0].set_xlabel('Nombre de bulletins')
for i, v in enumerate(bio.values):
    axes[0].text(v + 0.1, i, str(v), va='center', fontsize=9)

esp = df['nom_espece'].value_counts().head(10)
axes[1].barh(esp.index, esp.values, color=C_SECONDARY)
axes[1].set_title('Top 10 espèces', loc='left', fontweight='bold')
axes[1].invert_yaxis()
axes[1].set_xlabel('Nombre de bulletins')
for i, v in enumerate(esp.values):
    axes[1].text(v + 0.1, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3bis. Goulots d'étranglement et sous-ensembles stratégiques

Cette section explicite les populations réellement utiles pour la suite : calibration stricte, exploration loose, sous-ensemble confirmé et données PI faibles.

In [ ]:
summary_rows = [
    ('Tous bulletins', len(df)),
    ('PI uniquement', (df['type'] == 'PEST_IDENTIFICATION').sum()),
    ('Avec image', (df['n_images_found'].fillna(0) > 0).sum()),
    ('Avec date EXIF', df['datetime_observation'].notna().sum()),
    ('Avec coordonnées', df['latitude'].notna().sum()),
    ('Outdoor', df['outdoor'].sum()),
    ('phase2_ready strict', df['phase2_ready'].sum()),
    ('phase2_ready loose', df['phase2_ready_loose'].sum()),
    ('CONFIRMED ∩ strict', (df['status_confirmed'] & df['phase2_ready']).sum()),
    ('PI ∩ strict', ((df['type'] == 'PEST_IDENTIFICATION') & df['phase2_ready']).sum()),
    ('PI ∩ loose', ((df['type'] == 'PEST_IDENTIFICATION') & df['phase2_ready_loose']).sum()),
]
summary = pd.DataFrame(summary_rows, columns=['population', 'n'])
summary['pct_total'] = 100 * summary['n'] / len(df)
summary

In [ ]:
# Visualisation du goulot principal : coordonnées/jardin
funnel = summary[summary['population'].isin([
    'Tous bulletins', 'Avec image', 'Avec date EXIF', 'Avec coordonnées',
    'Outdoor', 'phase2_ready strict'
])]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(funnel['population'], funnel['n'], marker='o', linewidth=2.5, color=C_PRIMARY)
ax.fill_between(range(len(funnel)), funnel['n'], alpha=0.15, color=C_PRIMARY)
ax.set_title('Goulot d’étranglement de la phase 2', loc='left', fontweight='bold')
ax.set_ylabel('Nombre de bulletins')
ax.tick_params(axis='x', rotation=30)
for i, row in funnel.reset_index(drop=True).iterrows():
    ax.text(i, row['n'] + 2, f"{int(row['n'])}", ha='center')
plt.tight_layout()
plt.show()

## 3ter. Cohérence temporelle intra-bulletin

Si les photos d'un même bulletin sont prises dans une même session, le pivot bulletin est méthodologiquement justifié.

In [ ]:
span = df.loc[df['date_span_minutes'].notna(), ['bulletin_id', 'n_images_found', 'date_span_minutes',
                                                'datetime_observation', 'nom_espece', 'status']]
print(f"Médiane écart intra-bulletin : {span['date_span_minutes'].median():.2f} min")
print(f"Max écart intra-bulletin     : {span['date_span_minutes'].max():.2f} min")
print(f"> 1 heure                    : {(span['date_span_minutes'] > 60).sum()} bulletin(s)")
display(span.sort_values('date_span_minutes', ascending=False).head(10))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(span['date_span_minutes'].dropna(), bins=20, color=C_SECONDARY, edgecolor='white')
ax.axvline(60, color=C_WARNING, linestyle='--', label='1 h')
ax.set_title('Écart temporel entre images d’un même bulletin', loc='left', fontweight='bold')
ax.set_xlabel('minutes')
ax.set_ylabel('Nombre de bulletins')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Carte géographique des 31 bulletins (plotly)

Carte interactive : la couleur distingue les bulletins **phase2_ready strict** (méteo pleinement pertinente) des bulletins **WINDOW** (météo peu pertinente, conservés pour audit).

In [ ]:
df_map = df_ready_loose.copy()
df_map['ready_type'] = df_map['phase2_ready'].map({True: 'phase2_ready', False: 'WINDOW (loose)'})
df_map['label'] = df_map.apply(
    lambda r: f"#{int(r['bulletin_id'])} · {r['city'] or '?'}<br>"
              f"{r['nom_espece'] or '?'}<br>"
              f"Diagnostic: {(r['nom_bioagresseur'] or 'Non renseigné')}<br>"
              f"Status: {r['status']}<br>"
              f"{r['n_images']} image(s) · {r['plant_location']}",
    axis=1
)

fig = px.scatter_geo(
    df_map,
    lat='latitude', lon='longitude',
    color='ready_type',
    size='n_images',
    size_max=18,
    hover_name='label',
    hover_data={'latitude': False, 'longitude': False,
                'ready_type': False, 'n_images': False},
    color_discrete_map={
        'phase2_ready': C_PRIMARY,
        'WINDOW (loose)': C_ACCENT,
    },
    projection='mercator',
)
fig.update_geos(
    visible=True,
    resolution=50,
    showcountries=True, countrycolor='#aaa',
    showsubunits=True, subunitcolor='#ddd',
    lonaxis_range=[-5, 9],
    lataxis_range=[42, 51.5],
)
fig.update_layout(
    title=dict(
        text='<b>Distribution géographique des 31 bulletins phase2_ready_loose</b>',
        x=0.02, xanchor='left'
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.05,
        xanchor='center', x=0.5,
    ),
    height=600,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig.show()

## 5. Chronologie des observations (plotly)

Survol pour voir chaque bulletin individuellement.

In [ ]:
df_time = df_ready_loose.sort_values('datetime_observation').reset_index(drop=True)
df_time['y'] = range(len(df_time))
df_time['hover'] = df_time.apply(
    lambda r: f"#{int(r['bulletin_id'])} · {r['datetime_observation'].strftime('%Y-%m-%d %H:%M')}<br>"
              f"{r['nom_espece'] or '?'} · {r['city'] or '?'}<br>"
              f"Diagnostic: {(r['nom_bioagresseur'] or 'Non renseigné')}<br>"
              f"Status: {r['status']}<br>"
              f"Plant location: {r['plant_location']}",
    axis=1
)

fig = px.scatter(
    df_time,
    x='datetime_observation', y='y',
    color='plant_location',
    size='n_images', size_max=15,
    hover_name='hover',
    hover_data={'datetime_observation': False, 'y': False,
                'plant_location': False, 'n_images': False},
    color_discrete_sequence=[C_SECONDARY, C_PRIMARY, C_ACCENT, C_NEUTRAL],
)
fig.update_layout(
    title=dict(
        text='<b>Chronologie des 31 bulletins phase2_ready_loose</b>',
        x=0.02, xanchor='left'
    ),
    xaxis_title='Date d\'observation',
    yaxis_title='Bulletin (trié par date)',
    yaxis_showticklabels=False,
    height=480,
    plot_bgcolor='white',
    margin=dict(l=20, r=20, t=50, b=20),
)
fig.update_xaxes(showgrid=True, gridcolor='#eee')
fig.update_yaxes(showgrid=True, gridcolor='#eee')
fig.show()

print(f"Période : {df_time['datetime_observation'].min().date()} → {df_time['datetime_observation'].max().date()}")
print(f"Durée : {(df_time['datetime_observation'].max() - df_time['datetime_observation'].min()).days} jours")

## 6. Features météo : distributions (matplotlib)

Vue d'ensemble des 12 variables agrégées sur la fenêtre J-15 → J-1 sur les 31 bulletins.

In [ ]:
FEATURES = [
    ('temp_mean_15d',          '°C',    'Température moyenne'),
    ('temp_max_15d',           '°C',    'Température max'),
    ('temp_min_15d',           '°C',    'Température min'),
    ('rain_sum_15d',           'mm',    'Pluie cumulée'),
    ('rain_days_15d',          'j',     'Jours de pluie (≥1mm)'),
    ('max_dry_streak_15d',     'j',     'Plus longue série sèche'),
    ('humidity_mean_15d',      '%',     'Humidité moyenne'),
    ('high_humidity_days_15d', 'j',     'Jours HR ≥ 85%'),
    ('vpd_max_15d',            'kPa',   'VPD max'),
    ('dew_point_mean_15d',     '°C',    'Point de rosée'),
    ('hot_days_15d',           'j',     'Jours chauds (>28°C)'),
    ('soil_moisture_mean_15d', '',      'Humidité du sol (moy.)'),
]
available = [(c, u, t) for c, u, t in FEATURES if c in df_features.columns]
n = len(available)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 2.6))
axes = axes.flatten()

for i, (col, unit, title) in enumerate(available):
    ax = axes[i]
    data = df_features[col].dropna()
    
    ax.hist(data, bins=12, color=C_PRIMARY, alpha=0.7, edgecolor='white')
    
    # Statistiques descriptives
    median = data.median()
    ax.axvline(median, color=C_SECONDARY, linewidth=1.5, linestyle='--',
               alpha=0.8, label=f'médiane={median:.1f}')
    
    ax.set_title(f'{title}', fontsize=9, loc='left')
    ax.set_xlabel(unit, fontsize=8)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, loc='upper right', framealpha=0.9)
    ax.grid(alpha=0.2)

for j in range(n, len(axes)):
    axes[j].axis('off')

plt.suptitle('Distribution des features météo sur les 31 bulletins\n'
             '(trait vert = médiane)',
             fontsize=11, fontweight='bold', x=0.04, ha='left', y=1.01)
plt.tight_layout()
plt.show()

## 7. Heatmap bulletins × features (plotly)

Survol pour voir la valeur exacte de chaque cellule. Tri chronologique sur Y.

In [ ]:
feature_cols = [c for c, _, _ in FEATURES if c in df_features.columns]

# Tri chronologique
df_heat = df_features.sort_values('datetime_observation').reset_index(drop=True)
M = df_heat[feature_cols].copy()

# Min-max normalisation par feature pour l'affichage couleur
M_norm = (M - M.min()) / (M.max() - M.min())

# Y-labels avec date et bioagresseur si renseigné (sans mise en avant CONFIRMED)
ylabels = []
for _, row in df_heat.iterrows():
    date_str = row['datetime_observation'].strftime('%m-%d') if pd.notna(row['datetime_observation']) else '??'
    label = f"#{int(row['bulletin_id'])} ({date_str})"
    if pd.notna(row.get('nom_bioagresseur')):
        label += f" · {row['nom_bioagresseur']}"
    ylabels.append(label)

# X-labels lisibles
xlabels = [c.replace('_15d', '') for c in feature_cols]

# Texte de hover : valeurs originales (non normalisées)
hover_text = [[f"{ylabels[i]}<br>{xlabels[j]}: {M.iloc[i, j]:.2f}"
               for j in range(len(feature_cols))]
              for i in range(len(df_heat))]

fig = go.Figure(data=go.Heatmap(
    z=M_norm.values,
    x=xlabels,
    y=ylabels,
    text=hover_text,
    hoverinfo='text',
    colorscale='RdYlBu_r',
    colorbar=dict(title='Valeur<br>normalisée', tickfont=dict(size=9)),
))
fig.update_layout(
    title=dict(
        text='<b>Heatmap features météo · 31 bulletins triés par date</b><br>'
             '<sub>Bleu = bas, Rouge = haut · Normalisation min-max par feature</sub>',
        x=0.02, xanchor='left'
    ),
    xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=8)),
    height=720,
    margin=dict(l=180, r=20, t=80, b=80),
)
fig.show()

## 7bis. Scénarios météo provisoires avant `weather_rules.py`

Ces scénarios ne remplacent pas les règles agronomiques finales. Ils servent à inspecter les 31 bulletins météo et à vérifier que les features produisent des profils interprétables.

In [ ]:
def classify_simple_scenario(row):
    rain = row.get('rain_sum_15d', np.nan)
    rain_days = row.get('rain_days_15d', np.nan)
    hum = row.get('humidity_mean_15d', np.nan)
    temp = row.get('temp_mean_15d', np.nan)
    hot = row.get('hot_days_15d', np.nan)
    vpd = row.get('vpd_max_15d', np.nan)
    dry = row.get('max_dry_streak_15d', np.nan)

    if pd.isna(rain) or pd.isna(hum) or pd.isna(temp):
        return np.nan
    if (temp >= 18 and (hot >= 2 or vpd >= 2.2) and rain <= 10 and dry >= 5):
        return 'chaud_et_sec'
    if (rain >= 35 and rain_days >= 5 and hum >= 70):
        return 'humide_et_pluvieux'
    if (hum >= 70 and 10 <= temp <= 20):
        return 'doux_et_humide'
    if (rain <= 10 and dry >= 5):
        return 'sec'
    if (temp >= 18):
        return 'chaud'
    if (rain >= 25):
        return 'pluvieux'
    return 'modere'

df_features = df_features.copy()
df_features['scenario_simple'] = df_features.apply(classify_simple_scenario, axis=1)

scenario_counts = df_features['scenario_simple'].value_counts(dropna=False).rename_axis('scenario').reset_index(name='n')
display(scenario_counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(scenario_counts['scenario'].astype(str), scenario_counts['n'], color=C_ACCENT)
ax.invert_yaxis()
ax.set_title('Scénario météo provisoire — 31 bulletins', loc='left', fontweight='bold')
ax.set_xlabel('Nombre de bulletins')
for i, v in enumerate(scenario_counts['n']):
    ax.text(v + 0.2, i, str(v), va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Lecture croisée scénario × bioagresseur renseigné
cross_cols = ['bulletin_id', 'datetime_observation', 'nom_espece', 'nom_bioagresseur',
              'status', 'plant_location', 'weather_relevance',
              'scenario_simple', 'rain_sum_15d', 'humidity_mean_15d',
              'temp_mean_15d', 'vpd_max_15d', 'max_dry_streak_15d']
cross_cols = [c for c in cross_cols if c in df_features.columns]
df_features[cross_cols].sort_values(['scenario_simple', 'datetime_observation']).reset_index(drop=True)

## 7ter. Lecture agronomique exploratoire

Avant `weather_rules.py`, cette table permet de repérer visuellement des cas qui devraient être favorables ou défavorables à certaines familles de pathologies.

In [ ]:
# Indicateurs agronomiques simples pour préparer les règles
tmp = df_features.copy()
tmp['signal_fongique_humide'] = (
    (tmp['rain_sum_15d'] >= 25) &
    (tmp['rain_days_15d'] >= 4) &
    (tmp['humidity_mean_15d'] >= 70)
)
tmp['signal_acariens_sec'] = (
    (tmp['rain_sum_15d'] <= 10) &
    (tmp['max_dry_streak_15d'] >= 5) &
    ((tmp['vpd_max_15d'] >= 2.2) | (tmp['hot_days_15d'] >= 2))
)
tmp['signal_stress_froid'] = tmp['cold_days_15d'].fillna(0) >= 3

signals = tmp[['signal_fongique_humide', 'signal_acariens_sec', 'signal_stress_froid']].sum().rename('n').reset_index()
signals.columns = ['signal', 'n']
display(signals)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(signals['signal'], signals['n'], color=[C_SECONDARY, C_WARNING, C_NEUTRAL])
ax.invert_yaxis()
ax.set_title('Signaux météo agronomiques simples', loc='left', fontweight='bold')
ax.set_xlabel('Nombre de bulletins')
for i, v in enumerate(signals['n']):
    ax.text(v + 0.2, i, str(int(v)), va='center')
plt.tight_layout()
plt.show()

## 8. Tableau détaillé des 31 bulletins phase2_ready_loose

Liste complète, sans pré-sélection, triée par date.

In [ ]:
cols_display = [
    'bulletin_id', 'datetime_observation', 'nom_espece', 'nom_bioagresseur',
    'status', 'city', 'plant_location',
    'temp_mean_15d', 'rain_sum_15d', 'rain_days_15d',
    'humidity_mean_15d', 'vpd_max_15d',
    'soil_moisture_mean_15d', 'hot_days_15d', 'max_dry_streak_15d',
]
cols_display = [c for c in cols_display if c in df_ready_loose.columns]

df_display = df_ready_loose[cols_display].sort_values('datetime_observation').reset_index(drop=True)
df_display

## 9. Synthèse pour le rapport

In [ ]:
print('=' * 60)
print('SYNTHÈSE PHASE 2 — Export 12 mai 2026')
print('=' * 60)
print()
print('Volume de départ : 127 bulletins, 152 images')
print()
print(f'Couverture EXIF date         : {df["datetime_observation"].notna().sum():>4} / 127 ({100*df["datetime_observation"].notna().sum()/127:.1f}%)')
print(f'Avec jardin renseigné (GPS)  : {df["latitude"].notna().sum():>4} / 127 ({100*df["latitude"].notna().sum()/127:.1f}%)')
print(f'phase2_ready (calibration)   : {df["phase2_ready"].sum():>4} / 127 ({100*df["phase2_ready"].sum()/127:.1f}%)')
print(f'phase2_ready_loose (+WINDOW) : {df["phase2_ready_loose"].sum():>4} / 127 ({100*df["phase2_ready_loose"].sum()/127:.1f}%)')
print()
print('Plafond fonctionnel : la phase 2 ne s\'applique qu\'aux bulletins')
print('dont l\'utilisateur a configuré un jardin (26% du corpus).')
print()
print('Bioagresseurs renseignés sur les 31 phase2_ready_loose :')
bio_loose = df_ready_loose['nom_bioagresseur'].value_counts(dropna=False)
for bio, n in bio_loose.items():
    label = str(bio) if pd.notna(bio) else 'Non renseigné'
    print(f'  - {label:30s} {n}')
print()
print('Période couverte par les observations :')
print(f'  {df_features["datetime_observation"].min().date()} → {df_features["datetime_observation"].max().date()}')
print()
print('Profil météo dominant (printemps 2026) :')
print(f'  Température moyenne : {df_features["temp_mean_15d"].mean():.1f} °C')
print(f'  Pluie cumulée moy.  : {df_features["rain_sum_15d"].mean():.1f} mm / 15j')
print(f'  Humidité moy.       : {df_features["humidity_mean_15d"].mean():.1f} %')
print(f'  VPD max moy.        : {df_features["vpd_max_15d"].mean():.2f} kPa')
print(f'  Jours chauds (>28°) : {df_features["hot_days_15d"].sum()} (total)')

## 9bis. Phrases prêtes pour le rapport

Cette cellule génère un texte court à intégrer dans le rapport d'avancement phase 2.

In [ ]:
n_total = len(df)
n_exif = int(df['datetime_observation'].notna().sum())
n_coords = int(df['latitude'].notna().sum())
n_strict = int(df['phase2_ready'].sum())
n_loose = int(df['phase2_ready_loose'].sum())
span_med = df['date_span_minutes'].dropna().median()
span_gt1h = int((df['date_span_minutes'] > 60).sum())
n_features = int(df['rain_sum_15d'].notna().sum())

texte = (
    f"Sur les {n_total} bulletins officiels de l'export, {n_exif} disposent d'une date "
    f"d'observation exploitable issue des EXIF ou de l'agrégation des images. La limite "
    f"principale de la phase météo ne provient donc pas de la date, mais de la disponibilité "
    f"d'un jardin géolocalisé : {n_coords} bulletins disposent de coordonnées. Après filtrage "
    f"par pertinence météo extérieure, {n_strict} bulletins sont exploitables en mode strict "
    f"et {n_loose} en mode loose incluant les cas WINDOW.\n\n"
    f"La cohérence temporelle intra-bulletin est forte : l'écart médian entre images d'un "
    f"même bulletin est de {span_med:.1f} minute(s), et seulement {span_gt1h} bulletin(s) "
    f"présente(nt) un écart supérieur à une heure. Cela valide le choix méthodologique de "
    f"ramener les données au niveau du bulletin avant d'interroger la météo.\n\n"
    f"Les features météo ont été calculées pour {n_features} bulletins. Elles décrivent la "
    f"fenêtre J-15 à J-1 précédant l'observation, avec des indicateurs de pluie, humidité, "
    f"température, VPD, vent, bilan hydrique et humidité du sol. Ces variables constituent "
    f"l'entrée du module weather_rules.py, chargé d'estimer la compatibilité entre un "
    f"diagnostic candidat et le contexte environnemental."
)
print(texte)

---

## Prochaine étape : `weather_rules.py`

Maintenant qu'on a une vision complète et non biaisée des 31 bulletins, on peut passer à la classification :

- **Appliquer les règles à tous les bulletins phase2_ready** (sans préjuger du statut utilisateur)
- **Comparer le verdict des règles avec le bioagresseur renseigné** quand il existe — c'est un signal exploratoire, pas une vérité terrain
- **Identifier les pathologies dominantes du corpus qui ne sont pas couvertes** par les règles actuelles (Pucerons, Oïdium, etc.)